# Advanced projects A-E

Played matches only; scheduled fixtures are 'Coming soon' and never analysed as results. Source: PostgreSQL warehouse (`docker compose up -d`) and saved reports.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)


In [2]:
from fbplatform.analysis import advanced as A

## A. Tournament competitiveness index
Equal-weight mean of percentile ranks of four components; differences of a few hundredths are within noise (see the margin interval).

In [3]:
ci, sens = A.competitiveness()
ci[['season_id', 'competitiveness_index', 'mean_abs_margin', 'margin_ci_low', 'margin_ci_high']].round(3).head(12)

,season_id,competitiveness_index,mean_abs_margin,margin_ci_low,margin_ci_high
0,EURO-2004,0.929,1.214,0.799,1.655
1,EURO-2024,0.848,1.078,0.824,1.353
2,UNL-2020-21,0.786,1.166,1.000,1.346
3,EURO-2016,0.772,1.261,0.958,1.535
4,WC-2010,0.714,1.203,0.953,1.562
5,COPA-2011,0.714,1.091,0.659,1.542
6,COPA-2004,0.634,1.417,0.918,1.938
7,EURO-2020,0.625,1.314,0.990,1.627
8,WC-2002,0.607,1.250,0.969,1.594
9,UNL-2022-23,0.607,1.422,1.224,1.620


In [4]:
sens.round(3)

,spearman_full_vs_without_upset,spearman_full_vs_margin_only
0,0.919,0.712


## B. Team dominance

In [5]:
A.dominance()[['team_name', 'editions', 'matches', 'win_pct_shrunk', 'gd_pm', 'titles', 'dominance_score']].head(12).round(3)

,team_name,editions,matches,win_pct_shrunk,gd_pm,titles,dominance_score
0,Argentina,15,86,62.245,1.105,3.0,0.969
1,Spain,19,105,59.787,1.029,6.0,0.956
2,France,19,106,54.099,0.594,3.0,0.951
3,Brazil,16,83,56.726,1.084,4.0,0.910
4,Germany,19,100,47.959,0.690,1.0,0.908
5,Greece,10,50,49.592,0.300,1.0,0.849
6,Portugal,19,99,51.151,0.727,3.0,0.829
7,Norway,7,37,50.542,0.486,0.0,0.824
8,Colombia,12,60,53.936,0.583,1.0,0.823
9,Italy,16,84,45.484,0.429,2.0,0.795


## C. Player contribution index (position-aware, shrunk, ranked only for 900+ minutes)

In [6]:
A.player_contribution()[['full_name', 'position', 'minutes', 'goals', 'rate_shrunk_p90', 'pci_z_within_position', 'sample_size_flag']].head(12).round(3)

,full_name,position,minutes,goals,rate_shrunk_p90,pci_z_within_position,sample_size_flag
0,Denzel Dumfries,Defender,1314.0,3,0.179,4.815,ok
1,Raphaël Adelino José Guerreiro,Defender,1066.4,2,0.179,4.804,ok
2,Daniel Muñoz Mejía,Defender,1357.1,4,0.176,4.697,ok
3,Antoine Griezmann,Midfielder,2517.2,11,0.387,4.475,ok
4,Thomas Meunier,Defender,1593.1,2,0.159,4.121,ok
5,Philippe Coutinho Correia,Midfielder,1498.5,7,0.345,3.776,ok
6,Luke Shaw,Defender,1232.9,1,0.144,3.591,ok
7,Kylian Mbappé Lottin,Forward,2806.7,23,0.692,3.559,ok
8,Joakim Mæhle,Defender,1161.8,2,0.127,3.007,ok
9,Harry Maguire,Defender,1646.3,2,0.120,2.783,ok


## D. Trend forecasting: EXPLORATORY, not reliable predictions

In [7]:
bt, fc = A.forecast()
bt.round(3)

,competition,method,editions,backtest_origins,mae_goals_per_match
0,WC,naive_mean,7,4,0.271
1,WC,linear_trend,7,4,0.222
2,WC,holt,7,4,0.248
3,EURO,naive_mean,7,4,0.214
4,EURO,linear_trend,7,4,0.278
5,EURO,holt,7,4,0.278
6,COPA,naive_mean,9,6,0.348
7,COPA,linear_trend,9,6,0.563
8,COPA,holt,9,6,0.562
9,UNL,naive_mean,5,2,0.137


In [8]:
fc[['competition', 'next_edition', 'method_selected_by_backtest', 'forecast_goals_per_match', 'pi_low', 'pi_high', 'status']].round(2)

,competition,next_edition,method_selected_by_backtest,forecast_goals_per_match,pi_low,pi_high,status
0,WC,2030,linear_trend,2.90,2.61,3.19,EXPLORATORY - not a reliable prediction
1,EURO,2028,naive_mean,2.49,2.03,2.95,EXPLORATORY - not a reliable prediction
2,COPA,2028,naive_mean,2.62,1.79,3.44,EXPLORATORY - not a reliable prediction
3,UNL,2028,naive_mean,2.51,2.16,2.85,EXPLORATORY - not a reliable prediction


## E. Head-to-head tool

In [9]:
h = A.head_to_head('Argentina', 'Brazil')
print(h.team_a, 'vs', h.team_b, '| meetings', h.meetings, '| W-D-L', h.wins_a, h.draws, h.wins_b, '| goals', h.goals_a, h.goals_b)
for c in h.caveats: print('-', c)
h.by_competition

Argentina vs Brazil | meetings 4 | W-D-L 1 1 2 | goals 3 7
- Meetings span different competitions, stages, venues and squads - treat them as context, not as a prediction.
- Only matches inside the four collected competitions are included (no friendlies or qualifiers), so the record is partial.
- Result = recorded score (extra time included, shoot-out excluded); shoot-out winners are listed separately.


,competition_id,meetings,wins_a,draws,wins_b,goals_a,goals_b
0,COPA,4,1,1,2,3,7


In [10]:
h.recent

,match_date,season_id,stage_name,home_team_name,home_score,away_score,away_team_name,shootout_winner
672,2004-07-25,COPA-2004,Final,Argentina,2,2,Brazil,Brazil
19,2007-07-15,COPA-2007,Final,Brazil,3,0,Argentina,None
1501,2019-07-02,COPA-2019,Semi-finals,Brazil,2,0,Argentina,None
1027,2021-07-10,COPA-2021,Final,Argentina,1,0,Brazil,None
